In [1]:
import pandas as pd
import os

folder = "data"

tables = {
    "begin_inventory": pd.read_csv(os.path.join(folder, "begin_inventory.csv")),
    "end_inventory": pd.read_csv(os.path.join(folder, "end_inventory.csv")),
    "purchase_prices": pd.read_csv(os.path.join(folder, "purchase_prices.csv")),
    "purchases": pd.read_csv(os.path.join(folder, "purchases.csv")),
    "sales": pd.read_csv(os.path.join(folder, "sales.csv")),
    "vendor_invoice": pd.read_csv(os.path.join(folder, "vendor_invoice.csv"))
}

In [2]:
#Checking missing values
for name, df in tables.items():
    print(f"\n{name}")
    print(df.isnull().sum())


begin_inventory
InventoryId    0
Store          0
City           0
Brand          0
Description    0
Size           0
onHand         0
Price          0
startDate      0
dtype: int64

end_inventory
InventoryId       0
Store             0
City           1284
Brand             0
Description       0
Size              0
onHand            0
Price             0
endDate           0
dtype: int64

purchase_prices
Brand             0
Description       1
Price             0
Size              1
Volume            1
Classification    0
PurchasePrice     0
VendorNumber      0
VendorName        0
dtype: int64

purchases
InventoryId       0
Store             0
Brand             0
Description       0
Size              3
VendorNumber      0
VendorName        0
PONumber          0
PODate            0
ReceivingDate     0
InvoiceDate       0
PayDate           0
PurchasePrice     0
Quantity          0
Dollars           0
Classification    0
dtype: int64

sales
InventoryId       0
Store             0
Brand   

In [3]:
#replacing NULL vlaues with Unknown
tables["end_inventory"][tables["end_inventory"]["City"].isna()]
tables["end_inventory"]["City"] = tables["end_inventory"]["City"].fillna("Unknown")

In [4]:
#Since it had description , size , volume NULL and price 0. Removed it
tables["purchase_prices"][tables["purchase_prices"].isna().any(axis=1)]
tables["purchase_prices"] = tables["purchase_prices"].drop(index=7915)

In [5]:
#Checking the NULL rows 
tables["purchases"][tables["purchases"]["Size"].isna()]

,InventoryId,Store,Brand,Description,Size,VendorNumber,VendorName,PONumber,PODate,ReceivingDate,InvoiceDate,PayDate,PurchasePrice,Quantity,Dollars,Classification
1109668,34_PITMERDEN_3121,34,3121,Pinnacle Rainbow Sherbet,NaN,12546,JIM BEAM BRANDS COMPANY,10938,2024-06-27,2024-07-04,2024-07-13,2024-08-16,6.93,7,48.51,1
1112426,34_PITMERDEN_5678,34,5678,Skinnygirl Pina Colada,NaN,12546,JIM BEAM BRANDS COMPANY,10938,2024-06-27,2024-07-09,2024-07-13,2024-08-16,6.93,6,41.58,1
1116302,39_EASTHALLOW_15365,39,15365,Alabaster 07 Tinta de Toro,NaN,9552,M S WALKER INC,10972,2024-06-29,2024-07-07,2024-07-13,2024-08-21,91.83,1,91.83,2


In [6]:
#Checking size of each brand 
tables["purchase_prices"][
    tables["purchase_prices"]["Brand"].isin([3121, 5678, 15365])
][["Brand", "Description", "Size"]]


,Brand,Description,Size
572,3121,Pinnacle Rainbow Sherbet,750mL
1305,5678,Skinnygirl Pina Colada,750mL
6933,15365,Alabaster 07 Tinta de Toro,750mL


In [7]:
#replacing size 
mask = (
    tables["purchases"]["Brand"].isin([3121, 5678, 15365]) &
    tables["purchases"]["Description"].isin([
        "Pinnacle Rainbow Sherbet",
        "Skinnygirl Pina Colada",
        "Alabaster 07 Tinta de Toro"
    ])
)

tables["purchases"].loc[mask, "Size"] = "750mL"

In [8]:
#Count of NUll values
tables["vendor_invoice"]["Approval"].value_counts(dropna=False)

Approval
NaN               5169
Frank Delahunt     374
Name: count, dtype: int64

In [9]:
#checking null rows
tables["vendor_invoice"][
    tables["vendor_invoice"]["Approval"].isna()
].head(10)

,VendorNumber,VendorName,InvoiceDate,PONumber,PODate,PayDate,Quantity,Dollars,Freight,Approval
0,105,ALTAMAR BRANDS LLC,2024-01-04,8124,2023-12-21,2024-02-16,6,214.26,3.47,NaN
1,4466,AMERICAN VINTAGE BEVERAGE,2024-01-07,8137,2023-12-22,2024-02-21,15,140.55,8.57,NaN
2,388,ATLANTIC IMPORTING COMPANY,2024-01-09,8169,2023-12-24,2024-02-16,5,106.60,4.61,NaN
3,480,BACARDI USA INC,2024-01-12,8106,2023-12-20,2024-02-05,10100,137483.78,2935.20,NaN
4,516,BANFI PRODUCTS CORP,2024-01-07,8170,2023-12-24,2024-02-12,1935,15527.25,429.20,NaN
5,2396,BLACK PRINCE DISTILLERY INC,2024-01-08,8191,2023-12-25,2024-02-06,23,234.83,2.30,NaN
6,1128,BROWN-FORMAN CORP,2024-01-09,8150,2023-12-23,2024-02-19,4684,65403.57,1808.77,NaN
7,1189,BULLY BOY DISTILLERS,2024-01-09,8171,2023-12-24,2024-02-04,6,132.30,5.29,NaN
8,1273,CALEDONIA SPIRITS INC,2024-01-06,8172,2023-12-24,2024-02-15,5,146.80,15.53,NaN
9,11567,CAMPARI AMERICA,2024-01-06,8151,2023-12-23,2024-02-20,1321,12039.71,398.71,NaN


In [10]:
#replacing them with pending 
tables["vendor_invoice"]["Approval"] = (
    tables["vendor_invoice"]["Approval"].fillna("Pending")
)

In [11]:
#Checking duplicate rows
for name, df in tables.items():
    print(f"{name}: {df.duplicated().sum()} duplicate rows")

begin_inventory: 0 duplicate rows
end_inventory: 0 duplicate rows
purchase_prices: 0 duplicate rows
purchases: 0 duplicate rows
sales: 0 duplicate rows
vendor_invoice: 0 duplicate rows


In [12]:
#Removing extra spaces
for name, df in tables.items():
    df.columns = df.columns.str.strip()

    for col in df.select_dtypes(include="object"):
        df[col] = df[col].str.strip()

In [13]:
#Fixing data types
for name, df in tables.items():

    for col in df.columns:

        if "date" in col.lower():
            df[col] = pd.to_datetime(df[col], errors="coerce")

In [14]:
#Checking negative values
for name, df in tables.items():

    numeric_cols = df.select_dtypes(include="number").columns

    for col in numeric_cols:
        print(name, col, (df[col] < 0).sum())

begin_inventory Store 0
begin_inventory Brand 0
begin_inventory onHand 0
begin_inventory Price 0
end_inventory Store 0
end_inventory Brand 0
end_inventory onHand 0
end_inventory Price 0
purchase_prices Brand 0
purchase_prices Price 0
purchase_prices Classification 0
purchase_prices PurchasePrice 0
purchase_prices VendorNumber 0
purchases Store 0
purchases Brand 0
purchases VendorNumber 0
purchases PONumber 0
purchases PurchasePrice 0
purchases Quantity 0
purchases Dollars 0
purchases Classification 0
sales Store 0
sales Brand 0
sales SalesQuantity 0
sales SalesDollars 0
sales SalesPrice 0
sales Volume 0
sales Classification 0
sales ExciseTax 0
sales VendorNo 0
vendor_invoice VendorNumber 0
vendor_invoice PONumber 0
vendor_invoice Quantity 0
vendor_invoice Dollars 0
vendor_invoice Freight 0


In [15]:
#Checking unique values
for name, df in tables.items():

    for col in df.select_dtypes(include="object"):
        print(f"\n{name} - {col}")
        print(df[col].value_counts().head(10))


begin_inventory - InventoryId
InventoryId
1_HARDERSFIELD_58         1
56_BEGGAR'S HOLE_43809    1
56_BEGGAR'S HOLE_43225    1
56_BEGGAR'S HOLE_43245    1
56_BEGGAR'S HOLE_43308    1
56_BEGGAR'S HOLE_43362    1
56_BEGGAR'S HOLE_43365    1
56_BEGGAR'S HOLE_43434    1
56_BEGGAR'S HOLE_43553    1
56_BEGGAR'S HOLE_43590    1
Name: count, dtype: int64

begin_inventory - City
City
MOUNTMEND       14264
EANVERNESS      13011
DONCASTER       10356
HORNSEY          9997
GOULCREST        7930
PITMERDEN        6026
HARDERSFIELD     5395
DRY GULCH        4347
ARBINGTON        4199
WANBORNE         4128
Name: count, dtype: int64

begin_inventory - Description
Description
Jagermeister Liqueur       459
Bacardi Superior Rum       447
Capt Morgan Spiced Rum     405
Jack Daniels No 7 Black    404
Jim Beam                   378
Absolut 80 Proof           377
Jose Cuervo Especial       374
Southern Comfort           373
Kahlua                     371
Baileys Irish Cream        370
Name: count, dtype: int

In [16]:
!pip install pyodbc sqlalchemy

In [18]:
from sqlalchemy import create_engine
from urllib.parse import quote_plus
# SQL Server Connection

server = r"SUFILAPTOP\SQLEXPRESS"
database = "Inventory"

driver = quote_plus("ODBC Driver 17 for SQL Server")

engine = create_engine(
    f"mssql+pyodbc://@{server}/{database}?driver={driver}&trusted_connection=yes"
)
# Convert Date Columns

date_columns = {
    "begin_inventory": ["startDate"],
    "end_inventory": ["endDate"],
    "purchases": ["PODate", "ReceivingDate", "InvoiceDate", "PayDate"],
    "sales": ["SalesDate"],
    "vendor_invoice": ["InvoiceDate", "PODate", "PayDate"]
}

for table, cols in date_columns.items():
    for col in cols:
        if col in tables[table].columns:
            tables[table][col] = pd.to_datetime(
                tables[table][col],
                errors="coerce"
            )
# Upload Tables to SQL Server

for table_name, df in tables.items():

    print(f"Uploading {table_name}...")

    df.to_sql(
        name=table_name,
        con=engine,
        if_exists="replace",      # replace existing table
        index=False
    )

    print(f"{table_name} uploaded successfully.")
# Verify Upload


for table_name in tables.keys():

    print(f"\n{'='*60}")
    print(f"TABLE : {table_name}")
    print("="*60)

    query = f"SELECT TOP 5 * FROM {table_name}"

    result = pd.read_sql(query, engine)

    print(result)

print("\nAll tables uploaded successfully!")

Uploading begin_inventory...


C:\Users\mirsu\anaconda3\Lib\site-packages\pandas\io\sql.py:1648: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


begin_inventory uploaded successfully.
Uploading end_inventory...
end_inventory uploaded successfully.
Uploading purchase_prices...
purchase_prices uploaded successfully.
Uploading purchases...
purchases uploaded successfully.
Uploading sales...
sales uploaded successfully.
Uploading vendor_invoice...
vendor_invoice uploaded successfully.

TABLE : begin_inventory
         InventoryId  Store          City  Brand                  Description  \
0  1_HARDERSFIELD_58      1  HARDERSFIELD     58  Gekkeikan Black & Gold Sake   
1  1_HARDERSFIELD_60      1  HARDERSFIELD     60       Canadian Club 1858 VAP   
2  1_HARDERSFIELD_62      1  HARDERSFIELD     62     Herradura Silver Tequila   
3  1_HARDERSFIELD_63      1  HARDERSFIELD     63   Herradura Reposado Tequila   
4  1_HARDERSFIELD_72      1  HARDERSFIELD     72         No. 3 London Dry Gin   

    Size  onHand  Price  startDate  
0  750mL       8  12.99 2024-01-01  
1  750mL       7  10.99 2024-01-01  
2  750mL       6  36.99 2024-01-01  